# Tugas Individu - Deep Learning

| | |
|---|---|
| **Nama** | Nurul Marisa Clara Waldi |
| **NIM** | H071241015 |
| **Mata Kuliah** | Deep Learning |
| **Dataset** | Fashion-MNIST |

---

## Deskripsi Proyek

Notebook ini mengimplementasikan dan membandingkan **3 arsitektur deep learning** pada dataset **Fashion-MNIST** (10 kelas pakaian & aksesori):

1. **Custom CNN** — Convolutional Neural Network dengan 2 blok konvolusi
2. **Bidirectional GRU (RNN)** — Memproses gambar sebagai data sekuensial
3. **Transfer Learning ResNet-18** — Model pretrained ImageNet yang di-fine-tune

Semua model dievaluasi dengan metrik: Accuracy, Loss, Precision, Recall, F1-Score, dan Confusion Matrix.

## 1. Import Library & Konfigurasi

In [ ]:
import os
import time
import csv
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Konfigurasi
SEED       = 42
BATCH_SIZE = 64
EPOCHS     = 5
LR         = 0.001
N_TRAIN    = 15000
N_TEST     = 3000
DATA_DIR   = './data'
OUTPUT_DIR = './outputs'

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot',
]

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Deteksi perangkat
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Menggunakan: Apple Silicon MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Menggunakan: CUDA ({torch.cuda.get_device_name(0)})')
else:
    DEVICE = torch.device('cpu')
    print('Menggunakan: CPU')

print(f'PyTorch versi: {torch.__version__}')

## 2. Dataset Fashion-MNIST

**Fashion-MNIST** adalah dataset benchmark yang menggantikan MNIST klasik, terdiri dari:
- 70.000 gambar grayscale ukuran **28×28 piksel**
- **10 kelas**: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- Split: 60.000 train / 10.000 test

Untuk efisiensi eksperimen, digunakan **subset** 15.000 train dan 3.000 test.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform)
full_test = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform)

rng = np.random.RandomState(SEED)
train_idx = rng.choice(len(full_train), N_TRAIN, replace=False)
test_idx  = rng.choice(len(full_test),  N_TEST,  replace=False)

train_loader = DataLoader(Subset(full_train, train_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(Subset(full_test,  test_idx),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {N_TRAIN} samples | Test: {N_TEST} samples')

# Visualisasi contoh gambar
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Contoh Gambar Fashion-MNIST', fontsize=13, fontweight='bold')
images, labels = next(iter(train_loader))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Definisi Arsitektur Model

### 3.1 Custom CNN

CNN (Convolutional Neural Network) sangat efektif untuk data gambar karena kemampuannya mengekstrak fitur spasial secara hierarkis. Arsitektur yang digunakan:

```
Input (1×28×28)
  → Block 1: Conv2d(1→32) → BatchNorm → ReLU → MaxPool  → (32×14×14)
  → Block 2: Conv2d(32→64) → BatchNorm → ReLU → MaxPool  → (64×7×7)
  → Flatten → Linear(3136→128) → ReLU → Dropout(0.3) → Linear(128→10)
```

### 3.2 Bidirectional GRU (RNN)

GRU (Gated Recurrent Unit) memproses gambar sebagai sekuens: setiap baris piksel (28 baris) dianggap sebagai satu time step dengan 28 fitur.
Bidirectional memungkinkan model membaca urutan dari dua arah.

```
Input (1×28×28) → squeeze → (28×28)  [28 time steps, 28 features]
  → BiGRU(hidden=128, layers=2, dropout=0.2)
  → Output terakhir (256) → Linear(256→10)
```

### 3.3 Transfer Learning ResNet-18

ResNet-18 pretrained pada ImageNet. Dimodifikasi untuk input grayscale dan 10 kelas Fashion-MNIST:

```
ResNet-18 (pretrained ImageNet)
  → conv1: diubah ke 1-channel (grayscale)
  → fc   : Dropout(0.3) + Linear(512→10)
```

In [ ]:
# ---- 1. Custom CNN ----
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

# ---- 2. Bidirectional GRU ----
class ImageBiGRU(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=2, num_classes=10):
        super(ImageBiGRU, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            bidirectional=True, dropout=0.2,
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        x = x.squeeze(1)
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

# ---- 3. Transfer Learning ResNet-18 ----
class ResNet18_Transfer(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18_Transfer, self).__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
        self.model = base
    def forward(self, x):
        return self.model(x)

# Verifikasi parameter
for name, cls in [('CustomCNN', CustomCNN), ('ImageBiGRU', ImageBiGRU), ('ResNet18_Transfer', ResNet18_Transfer)]:
    m = cls()
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:25s}: {n_params:,} parameter')

## 4. Training & Evaluasi

Semua model dilatih dengan konfigurasi yang sama:
- **Optimizer**: Adam (lr=0.001)
- **Loss Function**: CrossEntropyLoss
- **Epochs**: 5
- **Batch Size**: 64

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += images.size(0)
    return total_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += images.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss/total, 100.0*correct/total, np.array(all_preds), np.array(all_labels)

print('Fungsi training dan evaluasi siap.')

In [ ]:
models_config = [
    (CustomCNN(),          'Custom CNN',         'Custom_CNN'),
    (ImageBiGRU(),        'BiGRU RNN',           'BiGRU_RNN'),
    (ResNet18_Transfer(), 'ResNet18 Transfer',   'ResNet18_Transfer'),
]

histories, model_names, csv_rows = [], [], []
all_results = {}  # simpan preds/labels untuk confusion matrix

for model, display_name, file_key in models_config:
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f'\n{"="*55}')
    print(f'  TRAINING: {display_name}')
    print(f'{"="*55}')
    print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>10} {"Val Loss":>10} {"Val Acc":>9}')
    print('-' * 55)

    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        vl, va, preds, labels = evaluate(model, test_loader, criterion, DEVICE)
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl);   history['val_acc'].append(va)
        print(f'{epoch:>6} {tl:>12.4f} {ta:>9.2f}% {vl:>10.4f} {va:>8.2f}%')

    train_time = time.time() - start
    f1 = f1_score(labels, preds, average='macro')

    histories.append(history)
    model_names.append(display_name)
    all_results[file_key] = (labels, preds, display_name)

    csv_rows.append({
        'Model': display_name,
        'Train Acc (%)': f'{history["train_acc"][-1]:.2f}',
        'Val Acc (%)':   f'{history["val_acc"][-1]:.2f}',
        'Val Loss':      f'{history["val_loss"][-1]:.4f}',
        'Macro F1-Score':f'{f1:.4f}',
        'Waktu Training (s)': f'{train_time:.1f}',
    })

    print(f'\n[Classification Report - {display_name}]')
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

## 5. Visualisasi Hasil

### 5.1 Perbandingan Accuracy & Loss per Epoch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perbandingan Model - Fashion-MNIST', fontsize=14, fontweight='bold')
colors = ['#2196F3', '#4CAF50', '#FF5722']
epochs_range = range(1, EPOCHS + 1)

for hist, name, c in zip(histories, model_names, colors):
    axes[0].plot(epochs_range, hist['val_acc'],  marker='o', label=name, color=c, linewidth=2)
    axes[1].plot(epochs_range, hist['val_loss'], marker='s', label=name, color=c, linewidth=2)

axes[0].set_title('Validation Accuracy (%)'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation Loss');         axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan!')

### 5.2 Confusion Matrix Tiap Model

In [ ]:
for file_key, (labels, preds, display_name) in all_results.items():
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'Confusion Matrix - {display_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Prediksi', fontsize=11)
    ax.set_ylabel('Label Aktual', fontsize=11)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/cm_{file_key}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'CM {display_name} disimpan!')

## 6. Ringkasan Hasil Evaluasi

In [ ]:
df_results = pd.DataFrame(csv_rows)
print('\n=== RINGKASAN HASIL EVALUASI ===')
display(df_results)

# Simpan ke CSV
df_results.to_csv(f'{OUTPUT_DIR}/metrics_summary.csv', index=False)
print(f'\nCSV disimpan di {OUTPUT_DIR}/metrics_summary.csv')

## 7. Kesimpulan

Dari eksperimen ini, dapat disimpulkan beberapa hal:

1. **Custom CNN** menunjukkan keunggulan dalam mengekstrak fitur spasial gambar berkat operasi konvolusi yang dirancang khusus untuk data 2D.

2. **Bidirectional GRU** meskipun tidak dirancang khusus untuk gambar, mampu menangkap pola sekuensial dari baris piksel, namun umumnya kurang efisien dibanding CNN untuk data gambar.

3. **Transfer Learning ResNet-18** memanfaatkan bobot pretrained dari ImageNet yang sangat besar, sehingga memiliki keunggulan dalam representasi fitur meski dilatih hanya sebentar.

Eksperimen ini menunjukkan pentingnya memilih arsitektur yang sesuai dengan karakteristik data dalam deep learning.